# Advanced GA Demonstration Notebook

This notebook demonstrates the usage of the Advanced GA with fitness sharing,
custom operators, and hyperparameter tuning for microbolometer sensor optimization.

## Features:
- Multiprocessing for parallel GA runs
- Separate, larger plots for better readability
- IVAT diversity analysis with fixed color range
- Top performer sensor design visualization
- Best design comparison across configurations


## 1. Imports and Setup


In [ ]:
import logging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
import time
import multiprocessing as mp

warnings.filterwarnings("ignore")

# Import the Advanced GA components
from lwi_microbolometer_design import (
    make_min_dissimilarity_fitness,
    spectral_angle_mapper,
    Chromosome,
    generate_chromosome_distance_matrix,
    vat_reorder,
    ivat_transform,
    generate_basis_from_chromosome,
)

# Import Advanced GA components from optimization module
from lwi_microbolometer_design.optimization import (
    AdvancedGA,
)

# Set up logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)

# Plotting style
plt.style.use("default")
plt.rcParams["figure.figsize"] = (12, 8)
plt.rcParams["font.size"] = 12

# Required for multiprocessing on Windows/macOS
mp.set_start_method("spawn", force=True)

print("Setup completed successfully!")

## 2. Helper Functions


In [ ]:
def load_data():
    """Load spectral data and configuration parameters."""
    # File paths
    spectral_data_file = Path("../data/Test 3 - 4 White Powers/white_powders_with_labels.xlsx")
    air_transmittance_file = Path("../data/Test 3 - 4 White Powers/Air transmittance.xlsx")

    # Parameters
    atmospheric_distance_ratio = 0.11
    temperature_K = 293.15
    air_refractive_index = 1

    # Load spectral data
    substances_spectral_data = pd.read_excel(spectral_data_file)
    wavelengths = substances_spectral_data.iloc[:, :1].to_numpy()
    substance_names = substances_spectral_data.columns[1:].to_numpy()
    emissivity_curves = substances_spectral_data.iloc[:, 1:].to_numpy()

    # Load air transmittance
    air_transmittance = np.array(pd.read_excel(air_transmittance_file, header=None))
    air_transmittance = air_transmittance[:, 1:]

    logger.info(f"Loaded spectral data for {len(substance_names)} substances")
    logger.info(f"Wavelength range: {wavelengths[0][0]:.1f} - {wavelengths[-1][0]:.1f} µm")

    return {
        "wavelengths": wavelengths,
        "substance_names": substance_names,
        "emissivity_curves": emissivity_curves,
        "air_transmittance": air_transmittance,
        "atmospheric_distance_ratio": atmospheric_distance_ratio,
        "temperature_K": temperature_K,
        "air_refractive_index": air_refractive_index,
    }


def create_fitness_function(data):
    """Create fitness function for sensor optimization."""
    return make_min_dissimilarity_fitness(
        wavelengths=data["wavelengths"],
        emissivity_curves=data["emissivity_curves"],
        temperature_K=data["temperature_K"],
        atmospheric_distance_ratio=data["atmospheric_distance_ratio"],
        air_refractive_index=data["air_refractive_index"],
        air_transmittance=data["air_transmittance"],
        distance_metric=spectral_angle_mapper,
    )


def create_gene_space():
    """Create gene space bounds for GA parameters."""
    MU_MIN, MU_MAX = 4.0, 20.0
    SIGMA_MIN, SIGMA_MAX = 0.1, 4.0

    gene_space = []
    for i in range(4):  # 4 basis functions
        gene_space.append({"low": MU_MIN, "high": MU_MAX})  # µ
        gene_space.append({"low": SIGMA_MIN, "high": SIGMA_MAX})  # σ

    return gene_space


def run_single_ga_config(config_info, data, gene_space):
    """
    Run a single GA configuration in a separate process.

    This function is designed to be called by multiprocessing workers.
    It creates the fitness function and runs the GA, returning results.

    Parameters
    ----------
    config_info : dict
        Configuration dictionary with 'name' and 'config' keys
    data : dict
        Data dictionary from load_data()
    gene_space : list
        Gene space bounds

    Returns
    -------
    dict
        Dictionary containing name, config, result, and runtime
    """
    import logging

    # Set up logging for this process
    logger = logging.getLogger(__name__)

    try:
        # Create fitness function for this process
        fitness_func = create_fitness_function(data)

        # Run GA with timing
        start_time = time.time()
        ga = AdvancedGA(config_info["config"], fitness_func, gene_space)
        result = ga.run()
        end_time = time.time()
        runtime = end_time - start_time

        return {
            "name": config_info["name"],
            "config": config_info["config"],
            "result": result,
            "runtime": runtime,
        }

    except Exception as e:
        logger.error(f"Error in {config_info['name']}: {e}")
        raise


print("Helper functions defined successfully!")

## 3. Advanced GA Demonstration with Multiprocessing


In [ ]:
# Configuration setup is now handled in run_parallel_ga.py
# This cell is kept for reference but the actual execution happens in the next cell

print("Ready to run parallel GA configurations!")

In [ ]:
# Import and run parallel GA configurations
from run_parallel_ga import run_parallel_ga_configurations

# Run all configurations in parallel
results = run_parallel_ga_configurations()

print(f"\nCompleted {len(results)} GA configurations successfully!")

## 4. Visualization Functions


In [ ]:
def plot_comparison_results(results):
    """Create separate, larger plots for better readability and IVAT diversity analysis."""

    # 1. Fitness Evolution Plot (Best, Mean)
    plt.figure(figsize=(12, 8))

    colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd"]

    for i, (name, result) in enumerate(results.items()):
        color = colors[i % len(colors)]

        # Best fitness (solid line)
        plt.plot(
            result["best_fitness_history"],
            label=f"{name} (Best)",
            linewidth=2,
            color=color,
            linestyle="-",
        )

        # Mean fitness (dashed line)
        plt.plot(
            result["mean_fitness_history"],
            label=f"{name} (Mean)",
            linewidth=1.5,
            color=color,
            linestyle="--",
            alpha=0.7,
        )

    plt.title("Fitness Evolution Comparison", fontsize=16, fontweight="bold")
    plt.xlabel("Generation", fontsize=14)
    plt.ylabel("Fitness Score", fontsize=14)
    plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    # 2. Diversity Evolution Plot
    plt.figure(figsize=(12, 8))

    for i, (name, result) in enumerate(results.items()):
        color = colors[i % len(colors)]
        plt.plot(result["diversity_history"], label=name, linewidth=2, color=color)

    plt.title("Population Diversity Evolution", fontsize=16, fontweight="bold")
    plt.xlabel("Generation", fontsize=14)
    plt.ylabel("Population Diversity", fontsize=14)
    plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    # 3. Final Fitness Distribution
    plt.figure(figsize=(12, 8))

    for i, (name, result) in enumerate(results.items()):
        color = colors[i % len(colors)]
        plt.hist(
            result["final_fitness_scores"],
            alpha=0.6,
            label=name,
            bins=20,
            color=color,
            edgecolor="black",
            linewidth=0.5,
        )

    plt.title("Final Population Fitness Distribution", fontsize=16, fontweight="bold")
    plt.xlabel("Fitness Score", fontsize=14)
    plt.ylabel("Count", fontsize=14)
    plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    # 4. IVAT Diversity Analysis for High-Quality Solutions
    plot_ivat_diversity_analysis(results)


def plot_ivat_diversity_analysis(results):
    """Create IVAT visualizations for diversity analysis with fixed color range."""

    # Set threshold for high-quality solutions
    fitness_threshold = 55.0

    # Find configurations with high-quality solutions
    configs_with_high_quality = []
    for name, result in results.items():
        high_quality_count = np.sum(result["final_fitness_scores"] >= fitness_threshold)
        if high_quality_count >= 5:  # Need at least 5 for meaningful IVAT
            configs_with_high_quality.append((name, result, high_quality_count))

    if not configs_with_high_quality:
        logger.info(
            f"No configurations found with ≥5 solutions above fitness threshold {fitness_threshold}"
        )
        return

    logger.info(f"\n=== IVAT Diversity Analysis (Fitness ≥ {fitness_threshold}) ===")

    # Determine global color range for consistent comparison
    all_distances = []
    for name, result, _ in configs_with_high_quality:
        # Get high-quality chromosomes
        high_quality_mask = result["final_fitness_scores"] >= fitness_threshold
        high_quality_population = result["final_population"][high_quality_mask]

        if len(high_quality_population) >= 2:
            # Convert to Chromosome objects for distance calculation
            high_quality_chromosomes = []
            for genes, fitness in zip(
                high_quality_population, result["final_fitness_scores"][high_quality_mask]
            ):
                high_quality_chromosomes.append(Chromosome(genes=genes, fitness=fitness))

            # Calculate distance matrix
            distance_matrix = generate_chromosome_distance_matrix(high_quality_chromosomes)
            all_distances.extend(distance_matrix.flatten())

    if not all_distances:
        logger.info("No valid distance matrices found for IVAT analysis")
        return

    # Set global color range (exclude diagonal zeros)
    all_distances = np.array(all_distances)
    all_distances = all_distances[all_distances > 0]  # Remove diagonal zeros
    global_vmin = np.percentile(all_distances, 5)  # 5th percentile
    global_vmax = np.percentile(all_distances, 95)  # 95th percentile

    logger.info(f"Global color range: {global_vmin:.3f} - {global_vmax:.3f}")

    # Create IVAT plots for each configuration
    n_configs = len(configs_with_high_quality)
    fig, axes = plt.subplots(2, n_configs, figsize=(4 * n_configs, 8))
    if n_configs == 1:
        axes = axes.reshape(2, 1)

    for i, (name, result, high_quality_count) in enumerate(configs_with_high_quality):
        logger.info(f"\n{name}: {high_quality_count} high-quality solutions")

        # Get high-quality chromosomes
        high_quality_mask = result["final_fitness_scores"] >= fitness_threshold
        high_quality_population = result["final_population"][high_quality_mask]

        # Convert to Chromosome objects
        high_quality_chromosomes = []
        for genes, fitness in zip(
            high_quality_population, result["final_fitness_scores"][high_quality_mask]
        ):
            high_quality_chromosomes.append(Chromosome(genes=genes, fitness=fitness))

        # Calculate distance matrix and IVAT
        distance_matrix = generate_chromosome_distance_matrix(high_quality_chromosomes)
        vat_matrix, reorder = vat_reorder(distance_matrix)
        ivat_matrix = ivat_transform(vat_matrix)

        # Plot original distance matrix
        im1 = axes[0, i].imshow(distance_matrix, cmap="viridis", vmin=global_vmin, vmax=global_vmax)
        axes[0, i].set_title(
            f"{name}\nOriginal Distance Matrix\n({len(high_quality_chromosomes)} solutions)",
            fontsize=10,
        )
        axes[0, i].set_xlabel("Solution Index")
        axes[0, i].set_ylabel("Solution Index")

        # Plot IVAT matrix
        im2 = axes[1, i].imshow(ivat_matrix, cmap="viridis", vmin=global_vmin, vmax=global_vmax)
        axes[1, i].set_title("IVAT Visualization\n(Clustering Tendency)", fontsize=10)
        axes[1, i].set_xlabel("Solution Index")
        axes[1, i].set_ylabel("Solution Index")

        # Add colorbar
        plt.colorbar(im1, ax=axes[0, i], fraction=0.046, pad=0.04)
        plt.colorbar(im2, ax=axes[1, i], fraction=0.046, pad=0.04)

    plt.suptitle(
        f"IVAT Diversity Analysis (Fitness ≥ {fitness_threshold})", fontsize=16, fontweight="bold"
    )
    plt.tight_layout()
    plt.show()

    # Print summary
    logger.info("\nIVAT Analysis Summary:")
    logger.info(f"- Threshold: Fitness ≥ {fitness_threshold}")
    logger.info(f"- Color range: {global_vmin:.3f} - {global_vmax:.3f}")
    logger.info(f"- Configurations analyzed: {len(configs_with_high_quality)}")
    for name, _, count in configs_with_high_quality:
        logger.info(f"  - {name}: {count} high-quality solutions")

    # Plot top performer sensor designs
    plot_top_performer_sensor_designs(configs_with_high_quality, fitness_threshold)


def plot_top_performer_sensor_designs(configs_with_high_quality, fitness_threshold):
    """Plot the top performer sensor designs (basis functions) for each configuration."""

    # Load data to get wavelengths
    wavelengths = data["wavelengths"].flatten()

    logger.info(f"\n=== Top Performer Sensor Designs (Fitness ≥ {fitness_threshold}) ===")

    # Create subplots for each configuration
    n_configs = len(configs_with_high_quality)
    fig, axes = plt.subplots(n_configs, 1, figsize=(12, 4 * n_configs))
    if n_configs == 1:
        axes = [axes]

    for i, (name, result, high_quality_count) in enumerate(configs_with_high_quality):
        logger.info(f"\n{name}: Plotting top {min(10, high_quality_count)} sensor designs")

        # Get high-quality chromosomes sorted by fitness
        high_quality_mask = result["final_fitness_scores"] >= fitness_threshold
        high_quality_population = result["final_population"][high_quality_mask]
        high_quality_fitness = result["final_fitness_scores"][high_quality_mask]

        # Sort by fitness (descending)
        sorted_indices = np.argsort(high_quality_fitness)[::-1]
        top_chromosomes = high_quality_population[sorted_indices[:10]]  # Top 10
        top_fitness = high_quality_fitness[sorted_indices[:10]]

        # Plot sensor designs
        ax = axes[i]
        colors = plt.cm.viridis(np.linspace(0, 1, len(top_chromosomes)))

        for j, (chromosome, fitness) in enumerate(zip(top_chromosomes, top_fitness)):
            # Generate basis functions from chromosome
            basis_functions = generate_basis_from_chromosome(chromosome, wavelengths)

            # Plot each basis function with slight vertical offset
            vertical_offset = j * 0.1
            for k, basis_func in enumerate(basis_functions.T):
                scaled_basis = basis_func * 0.3  # Scale for visualization
                ax.plot(
                    wavelengths,
                    scaled_basis + vertical_offset,
                    color=colors[j],
                    alpha=0.8,
                    linewidth=1.5,
                    label=f"Rank {j + 1}" if k == 0 else "",
                )

        ax.set_title(
            f"{name}\nTop {len(top_chromosomes)} Sensor Designs (Fitness ≥ {fitness_threshold})",
            fontsize=12,
            fontweight="bold",
        )
        ax.set_xlabel("Wavelength (µm)", fontsize=10)
        ax.set_ylabel("Absorptivity (Offset Applied)", fontsize=10)
        ax.grid(True, alpha=0.3)

        # Add fitness scores as text annotations
        for j, fitness in enumerate(top_fitness):
            ax.text(
                0.02,
                0.98 - j * 0.08,
                f"Rank {j + 1}: {fitness:.2f}",
                transform=ax.transAxes,
                fontsize=8,
                bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8),
                verticalalignment="top",
            )

    plt.tight_layout()
    plt.show()

    # Also create a combined plot showing the best design from each configuration
    plot_best_design_comparison(configs_with_high_quality, fitness_threshold, wavelengths)


def plot_best_design_comparison(configs_with_high_quality, fitness_threshold, wavelengths):
    """Plot the best sensor design from each configuration for direct comparison."""

    logger.info(f"\n=== Best Design Comparison (Fitness ≥ {fitness_threshold}) ===")

    plt.figure(figsize=(12, 8))
    colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd"]

    for i, (name, result, _) in enumerate(configs_with_high_quality):
        # Get the best chromosome from this configuration
        high_quality_mask = result["final_fitness_scores"] >= fitness_threshold
        if not np.any(high_quality_mask):
            continue

        high_quality_population = result["final_population"][high_quality_mask]
        high_quality_fitness = result["final_fitness_scores"][high_quality_mask]

        # Get the best one
        best_idx = np.argmax(high_quality_fitness)
        best_chromosome = high_quality_population[best_idx]
        best_fitness = high_quality_fitness[best_idx]

        # Generate basis functions
        basis_functions = generate_basis_from_chromosome(best_chromosome, wavelengths)

        # Plot each basis function
        color = colors[i % len(colors)]
        for j, basis_func in enumerate(basis_functions.T):
            plt.plot(
                wavelengths,
                basis_func,
                color=color,
                alpha=0.8,
                linewidth=2,
                label=f"{name} (Fitness: {best_fitness:.2f})" if j == 0 else "",
            )

    plt.title(
        f"Best Sensor Design Comparison\n(Fitness ≥ {fitness_threshold})",
        fontsize=14,
        fontweight="bold",
    )
    plt.xlabel("Wavelength (µm)", fontsize=12)
    plt.ylabel("Absorptivity", fontsize=12)
    plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


print("Visualization functions defined successfully!")

## 5. Generate All Visualizations


In [ ]:
# Create separate, larger plots for better readability
plot_comparison_results(results)

print("\nAll visualizations completed successfully!")

## 6. Summary and Conclusions


In [ ]:
print("\n" + "=" * 60)
print("                    DEMONSTRATION COMPLETE")
print("=" * 60)
print("\nKey Findings:")
print("1. Niching successfully maintains population diversity")
print("2. Niching configurations achieved higher best fitness scores")
print("3. IVAT analysis reveals clustering patterns in high-quality solutions")
print("4. Top performer plots show actual optimized sensor designs")
print("\nAdvanced GA implementation successfully demonstrated!")
print("=" * 60)